## Exploration and comparison of tasseled cap components generated from different coefficients

This notebook compares Tasseled cap wetness, greeness and brightness (TCW, TCG and TCB) derived from different sets of coefficients to see if there are significant differences in the output data. 

### Why do this work?

We currently use a single set of coefficients for generating TCW, TCG and TCB across all of the Landsat sensors (5, 7, 8 and 9). We use the coefficients of [Crist (1985)](https://www.sciencedirect.com/science/article/abs/pii/0034425785901026?via%3Dihub) to calculate the TC components from the DEA Landsat surface reflectance data ([see here](https://link.springer.com/content/pdf/10.1007/s13157-023-01682-7.pdf) for explanation of how TCW is generated to be used in WIT).

However, there are sensor-specific coefficients published and widely used for each Landsat sensor, and the use of a single set of coefficients to generate TC's for all sensors may introduce errors into our final products. A comparison between the TC's being created using the currently implemented coefficients, and those recommended in the wider literature, should be done to determine if our TC product needs to be updated to include sensor-specific coefficients.

### What this notebook does

- Loads DEA Landsat ARD data
- Uses `dea_tools` `calculate_indices` to calculate TCW, TCG and TCB using the currently implemented coefficients, and also the Landsat 8/9 coefficients from [Baig et al. 2014](https://www.tandfonline.com/doi/citedby/10.1080/2150704X.2014.915434?scroll=top&needAccess=true)
- **ALso now the TCT coefficients in the IEEE conference paper by Bex, Dale and Robert**
- `dea_tools` `bandindices.py` has been modified to include versions of the TC equations using the Landsat 8 coefficients.
- Calculates the difference between each TC component generated using the two sets of coefficients
- Generates figures to illustrate the differences between the TC outputs of the differenc coefficients
- Calculates correlation for the pairs of coefficients
- Genrates the RMSE, MAE and bias between the coefficients for each TC

### Still in progress:
- import wofs and generate other spectral indicies and use this extra information to see if there are larger discrepencies between the TC coefficients for particular land covers (currently trying using FC)
- scale up the area of analysis to cover a wider area and variety of land covers. Potentially using the collect training data moduel from dea-tools (Africa) or selecting subsets from the golden tiles, or randomly selected pixels across the continent. Currently the notebook runs for a small area in Victoria, so the results should be seen as preliminary and for assessing the notebook itself, not the TC's.
- Add hypothesis testing (done)
- Investigate the broader literature on what degree of pre-rpocessing is ususally applied to the input data. Our ARD data has several quality checks/ processes applied to it. If it is different enough from what is published, and it would have a significant impact on the resulting coefficients, there would be an argument for DEA to generate our own coefficients that are tailored to our ARD data (To discuss with Bex)



In [1]:
%pip install ultraplot xskillscore pypalettes -q
%pip install flox -q
#%pip install -U odc-stats -q


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Setup

- Import packages
- start up dask
- initiate datacube
- establish base query (will be used for loading several datasets through the notebook)
- load data for landsat sensor

In [2]:
import datacube
import os
import pandas as pd
import numpy as np
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
import ultraplot as uplt
import xskillscore
import cartopy.crs as ccrs
from matplotlib.dates import MonthLocator, DateFormatter
from datacube.utils import masking
from scipy.stats import linregress
import calendar
import pypalettes
import warnings
import time
import odc.geo.xr
from odc.geo.geom import Geometry, CRS
from shapely.geometry import box

import flox
from odc.algo._percentile import xr_quantile_bands

import sys

sys.path.insert(1, ".../Tools")
from dea_tools.datahandling import load_ard
from dea_tools.plotting import rgb, display_map
from dea_tools.dask import create_local_dask_cluster
from dea_tools.bandindices import calculate_indices
from dea_tools.spatial import add_geobox

warnings.filterwarnings("ignore")


In [3]:
create_local_dask_cluster()


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/40157/status,
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/40157/status,Workers: 1
Total threads: 15,Total memory: 117.21 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:45375,Workers: 1
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/40157/status,Total threads: 15
Started: Just now,Total memory: 117.21 GiB
Comm: tcp://127.0.0.1:44049,Total threads: 15
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/44767/status,Memory: 117.21 GiB
Nanny: tcp://127.0.0.1:39741,


2025-07-16 01:59:33,719 - distributed.nanny - WARNING - Restarting worker


In [4]:
dc = datacube.Datacube(app="TCP_initial_exploration")


In [5]:
save_dir = "./dev/development_notebooks_JAG/01_projects/tassel_cap_exploration/"


In [6]:
sample_points_dir = "sample_points_abares.gpkg"

sample_points_gpd = gpd.read_file(sample_points_dir)

In [21]:
sample_points_gpd.head()


,spatial_ref,class,geometry
0,3577,400,POINT (1089915.000 -3959205.000)
1,3577,400,POINT (1110495.000 -3981225.000)
2,3577,400,POINT (1085865.000 -3967605.000)
3,3577,400,POINT (1071555.000 -3967725.000)
4,3577,400,POINT (1093755.000 -3980865.000)


In [8]:
sample_points_gpd.total_bounds

array([ 1056225., -4031565.,  1151865., -3936405.])

In [9]:
xmin, ymin, xmax, ymax = sample_points_gpd.total_bounds
polygon = box(xmin, ymin, xmax, ymax)

In [11]:
geom = Geometry(geom=polygon, crs=sample_points_gpd.crs)

In [12]:
resolution = (-30, 30)
measurements = [
    "nbart_blue",
    "nbart_green",
    "nbart_red",
    "nbart_nir",
    "nbart_swir_1",
    "nbart_swir_2",
    "oa_fmask",
    "oa_nbart_contiguity",
]

In [13]:
# set up baseline dc query. THis will be modified by functions as needed later on.
query = {
    "geopolygon":geom,
    "time": ("2024-01", "2024-12"),
    "resolution": resolution,
    "group_by": "solar_day",
    "output_crs": "EPSG:3577",
    "dask_chunks": {"time": 60, "x": 1024, "y": 1024},
}


### Load the Landsat ARD data 

We will use this to generate the TC's, rather than use the existing TC product. This way we know the same pre-processing has been applied to both sets of coefficients.

In [14]:
%%time

ds = load_ard(
    dc=dc,
    products=['ga_ls8c_ard_3', 'ga_ls9c_ard_3'],
    measurements = ["nbart_blue", "nbart_green", "nbart_red", "nbart_nir", "nbart_swir_1", "nbart_swir_2", "oa_fmask", "oa_nbart_contiguity"], #, "oa_fmask", "oa_nbart_contiguity"
    cloud_mask = 'fmask',
    mask_pixel_quality=True,
    mask_contiguity = True,
    **query
)


Finding datasets
    ga_ls8c_ard_3
    ga_ls9c_ard_3
Applying fmask pixel quality/cloud mask
Applying contiguity mask (oa_nbart_contiguity)
Returning 91 time steps as a dask array
CPU times: user 1.59 s, sys: 78.3 ms, total: 1.67 s
Wall time: 1.72 s


In [15]:
ds


<xarray.Dataset> Size: 24GB
Dimensions:              (time: 91, y: 3173, x: 3189)
Coordinates:
  * time                 (time) datetime64[ns] 728B 2024-01-01T00:09:22.12330...
  * y                    (y) float64 25kB -3.936e+06 -3.936e+06 ... -4.032e+06
  * x                    (x) float64 26kB 1.056e+06 1.056e+06 ... 1.152e+06
    spatial_ref          int32 4B 3577
Data variables:
    nbart_blue           (time, y, x) float32 4GB dask.array<chunksize=(60, 1024, 1024), meta=np.ndarray>
    nbart_green          (time, y, x) float32 4GB dask.array<chunksize=(60, 1024, 1024), meta=np.ndarray>
    nbart_red            (time, y, x) float32 4GB dask.array<chunksize=(60, 1024, 1024), meta=np.ndarray>
    nbart_nir            (time, y, x) float32 4GB dask.array<chunksize=(60, 1024, 1024), meta=np.ndarray>
    nbart_swir_1         (time, y, x) float32 4GB dask.array<chunksize=(60, 1024, 1024), meta=np.ndarray>
    nbart_swir_2         (time, y, x) float32 4GB dask.array<chunksize=(60, 1024, 1024), meta=np.ndarray>
    oa_fmask             (time, y, x) uint8 921MB dask.array<chunksize=(60, 1024, 1024), meta=np.ndarray>
    oa_nbart_contiguity  (time, y, x) uint8 921MB dask.array<chunksize=(60, 1024, 1024), meta=np.ndarray>
Attributes:
    crs:           EPSG:3577
    grid_mapping:  spatial_ref

## Use the sample points to select pixels from the datacube

## Calculate TC's and metrics

- use `bandindices` to calculate the TC's using the existing coefficients (called `TCW`, `TCG` and `TCB` in this notebook) and the potential alternative coefficients ( named with the TC + sensor e.g. `TCW_ls8ls9`)
- get the difference between each TC pair
- Calculate RMSE, MAE and Bias from the difference

In [16]:
"""
NOTE: for now, before running the index calculations, you need to go to the '...Tools' directory and run pip install to access local changes.
The bandindices.py file has had additional TC equations added to it in the branch. These are not part of the standard dea_tools package.
"""

ds_tc = calculate_indices(
    ds,
    index=[
        "TCW",
        "TCB",
        "TCG",
        "TCW_ls8",
        "TCB_ls8",
        "TCG_ls8",
        "TCW_DEA",
        "TCB_DEA",
        "TCG_DEA",
    ],
    drop=True,
    inplace=True,
    collection="ga_ls_3",
)


Dropping bands ['nbart_blue', 'nbart_green', 'nbart_red', 'nbart_nir', 'nbart_swir_1', 'nbart_swir_2', 'oa_fmask', 'oa_nbart_contiguity']


In [17]:
"""
This list of variable pairs is used when we make graphs later on.
TODO: if notebook is extended to multi-year, may need to edit this.
"""

variable_pairs = {
    "baig_l8": [("TCW", "TCW_ls8"), ("TCG", "TCG_ls8"), ("TCB", "TCB_ls8")],
    "DEA_l8": [("TCW", "TCW_DEA"), ("TCG", "TCG_DEA"), ("TCB", "TCB_DEA")],
}


In [18]:
%%time

#make a new dataset to hold the metrics as they have different dims to the ds_tc (no x,y)
ds_metrics = xr.Dataset(coords={'time': ds_tc.coords['time']})
ds_metrics_monthly = xr.Dataset(coords={'time': ds_tc.coords['time'], 'month': ds_tc.coords['time.month']})

ds_metrics.attrs = ds_tc.attrs.copy()
ds_metrics_monthly.attrs = ds_tc.attrs.copy()

for j, group in variable_pairs.items():
    print(group)
    for i, (var1, var2) in enumerate(group):
        diff = ds_tc[var1] - ds_tc[var2]
        diff_month = diff.groupby('time.month')
        
        ds_tc[f'{var2}_diff'] = diff

        ds_metrics[f'{var2}_RMSE'] = np.sqrt((diff ** 2).mean(dim=['x', 'y']))
        ds_metrics[f'{var2}_MAE'] = np.abs(diff).mean(dim=['x', 'y'])
        ds_metrics[f'{var2}_bias'] = diff.mean(dim=['x', 'y'])
        
        ds_metrics_monthly[f'{var2}_RMSE'] = diff_month.map(lambda x: np.sqrt((x ** 2).mean(dim=['x', 'y'])))
        


[('TCW', 'TCW_ls8'), ('TCG', 'TCG_ls8'), ('TCB', 'TCB_ls8')]
[('TCW', 'TCW_DEA'), ('TCG', 'TCG_DEA'), ('TCB', 'TCB_DEA')]
CPU times: user 848 ms, sys: 18.1 ms, total: 866 ms
Wall time: 848 ms


In [19]:
ds_metrics


<xarray.Dataset> Size: 7kB
Dimensions:       (time: 91)
Coordinates:
  * time          (time) datetime64[ns] 728B 2024-01-01T00:09:22.123303 ... 2...
    spatial_ref   int32 4B 3577
Data variables: (12/18)
    TCW_ls8_RMSE  (time) float32 364B dask.array<chunksize=(60,), meta=np.ndarray>
    TCW_ls8_MAE   (time) float32 364B dask.array<chunksize=(60,), meta=np.ndarray>
    TCW_ls8_bias  (time) float32 364B dask.array<chunksize=(60,), meta=np.ndarray>
    TCG_ls8_RMSE  (time) float32 364B dask.array<chunksize=(60,), meta=np.ndarray>
    TCG_ls8_MAE   (time) float32 364B dask.array<chunksize=(60,), meta=np.ndarray>
    TCG_ls8_bias  (time) float32 364B dask.array<chunksize=(60,), meta=np.ndarray>
    ...            ...
    TCG_DEA_RMSE  (time) float32 364B dask.array<chunksize=(60,), meta=np.ndarray>
    TCG_DEA_MAE   (time) float32 364B dask.array<chunksize=(60,), meta=np.ndarray>
    TCG_DEA_bias  (time) float32 364B dask.array<chunksize=(60,), meta=np.ndarray>
    TCB_DEA_RMSE  (time) float32 364B dask.array<chunksize=(60,), meta=np.ndarray>
    TCB_DEA_MAE   (time) float32 364B dask.array<chunksize=(60,), meta=np.ndarray>
    TCB_DEA_bias  (time) float32 364B dask.array<chunksize=(60,), meta=np.ndarray>
Attributes:
    crs:           EPSG:3577
    grid_mapping:  spatial_ref

In [20]:
%%time

#load the datasets into memory now so graphs process faster and I can make modifications to the visualisations.
# grab all of the 'loading' operations and put them in one cell so that it is easier to track.

ds_metrics.load()
#ds_metrics_monthly.load()


/env/lib/python3.10/site-packages/rasterio/warp.py:387: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dest = _reproject(


KeyboardInterrupt: 

2025-07-16 01:59:32,713 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
Traceback (most recent call last):
  File "/env/lib/python3.10/site-packages/psutil/_common.py", line 502, in wrapper
    ret = self._cache[fun]
KeyError: <function Process._parse_stat_file at 0x7fac0e1ddc60>

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/env/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/env/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/env/lib/python3.10/site-packages/distributed/process.py", line 202, in _run
    target(*args, **kwargs)
  File "/env/lib/python3.10/site-packages/distributed/nanny.py", line 1022, in _run
    asyncio_run(run(), loop_factory=get_loop_factory())
  File "/env/lib/python3.10/site-packages/distributed/compatibility.py", line 239, in as

In [ ]:
ds_tc["time"] = pd.to_datetime(ds_tc["time"].values)
monthly_groups = ds_tc.groupby("time.month")

# for month, group in monthly_groups:
#     print(f'month: {month}')
#     print(group)
#     print('\n')


### Generate masks for grouping data by broad land types

Using wofs and fractional cover, create mask arrays that will be applied to the TC's generated from the coefficients. 

This is to try and see if, in the result that there are differences between the TC coefficients, if those differences are related to particular land covers or conditions.

In [ ]:
"""
NOTE: this is a VERY rough proxy and should be replaced with a better set of thresholds to get actual groups of bare soil,
consistently vegetated areas (e.g native vegetation or forests) and dynamic agricultural regions.
"""

fc_mask = 40

ds_fc = dc.load(
    product="ga_ls_fc_pc_cyear_3",
    measurements=["pv_pc_50", "bs_pc_50", "npv_pc_50"],
    # dask_chunks={"x": 1024, "y": 1024},
    **query
)

ds_fc = masking.mask_invalid_data(ds_fc)


In [ ]:
wofs_mask = 0.4  # set the threshold for classifying 'wet' pixels

ds_wofs = dc.load(
    product="ga_ls_wo_fq_cyear_3",
    measurements="frequency",
    # dask_chunks={"x": 1024, "y": 1024},
    **query
)

ds_wof = masking.mask_invalid_data(ds_wofs)


In [ ]:
wofs_mask = (ds_wofs["frequency"] > wofs_mask).astype(int)

green_mask = (ds_fc["pv_pc_50"] > fc_mask) & (wofs_mask == 0).astype(int)
bare_mask = (ds_fc["bs_pc_50"] > fc_mask) & (wofs_mask == 0).astype(int)
non_green_mask = (ds_fc["npv_pc_50"] > fc_mask) & (wofs_mask == 0).astype(int)


The masks created earlier have a single timestep as they are created from annual products. 
They are broadcast across all of the timesteps for the xarray Dataset containing the TC's

Also, the dask chunks are different after boradcasting the time dimension, so rechunk to try and make the dataset more efficient

In [ ]:
# Step 3: Expand dims to match ds_tc time dimension
ds_tc["wofs_mask"] = wofs_mask.isel(time=0).expand_dims(time=ds_tc.time)
ds_tc["pv_fc_mask"] = green_mask.isel(time=0).expand_dims(time=ds_tc.time)
ds_tc["bs_fc_mask"] = bare_mask.isel(time=0).expand_dims(time=ds_tc.time)
ds_tc["npv_fc_mask"] = non_green_mask.isel(time=0).expand_dims(time=ds_tc.time)

chunks = {"time": 5}
ds_tc.chunk(chunks)


### Plot the differences between the TC's generated from the two sets of coefficients

- Create masks based on wofs (to see 'wet' pixels) and fractional cover (to see pixels that are likely to be green veg, non-green veg and bare earth)
- NOTE: These masks are rough approximations, and are here as a first pass/ proof of concept. The actual threshold values, or even the data source for making the masks, should be considered more carefully in the final report.


In [ ]:
all_values = xr.DataArray(
    data=np.ones_like(ds_tc["pv_fc_mask"], dtype=bool),
    coords=ds_tc["pv_fc_mask"].coords,
    dims=ds_tc["pv_fc_mask"].dims,
)

class_masks = {
    " ": all_values,
    "wofs_mask": ds_tc["wofs_mask"],
    f"fc pv > {fc_mask}%": ds_tc["pv_fc_mask"],
    f"bs pv > {fc_mask}%": ds_tc["bs_fc_mask"],
    f"npv pv > {fc_mask}%": ds_tc["npv_fc_mask"],
}


In [ ]:
%%time
ds_diff_ls8 = np.stack([ds_tc['TCW_ls8_diff'].values.flatten(), ds_tc['TCG_ls8_diff'].values.flatten(), ds_tc['TCB_ls8_diff'].values.flatten()], axis=1)
ds_diff_ls8.shape


In [ ]:
%%time
ds_diff_dea = np.stack([ds_tc['TCW_DEA_diff'].values.flatten(), ds_tc['TCG_DEA_diff'].values.flatten(), ds_tc['TCB_DEA_diff'].values.flatten()], axis=1)
ds_diff_dea.shape


## Generate plots

In [ ]:
%%time

fig, ax = uplt.subplots(refwidth=8, refaspect=(8,4))
ax.format(suptitle='Histogram of differences between the currently implemented coefficients (Crist 1985) and the Landsat-8 coefficients (Baig et al 2014) for TCW, TCG and TCB', xlabel='Difference', ylabel='Frequency')
res = ax.hist(
    ds_diff_ls8,
    bins=100,
    histtype='stepfilled',
    filled=True,
    alpha=0.6,
    edgecolor='k',
    cycle=('indigo4', 'green4', 'yellow4'),
    labels=list(['TCW', 'TCG', 'TCB']),
    legend='ul'
    
)
ax.format(xlim=(-0.3, 0.2))

fig.savefig(os.path.join(save_dir, "histogram_tc_diff_baig.png"), dpi=100, bbox_inches='tight')
plt.close(fig)


In [ ]:

%%time

fig, ax = uplt.subplots(refwidth=8, refaspect=(8,4))
ax.format(suptitle='Histogram of differences between the currently implemented coefficients (Crist 1985) and the DEA IEEE paper coefficients for TCW, TCG and TCB', xlabel='Difference', ylabel='Frequency')
res = ax.hist(
    ds_diff_dea,
    bins=100,
    histtype='stepfilled',
    filled=True,
    alpha=0.6,
    edgecolor='k',
    cycle=('indigo4', 'green4', 'yellow4'),
    labels=list(['TCW', 'TCG', 'TCB']),
    legend='ul'
    
)
ax.format(xlim=(-0.3, 0.2))

fig.savefig(os.path.join(save_dir, "histogram_tc_diff_dea.png"), dpi=100, bbox_inches='tight')
plt.close(fig)


In [ ]:
%%time

for j, group in variable_pairs.items():
    print(j)
    fig, axs = plt.subplots(nrows=3, ncols=1, figsize=(6,9), layout='constrained')
    for i, (var1, var2) in enumerate(group):
        ax=axs[i]

        rmse = ds_metrics[f'{var2}_RMSE'].values.flatten()
        mae = ds_metrics[f'{var2}_MAE'].values.flatten()
        bias = ds_metrics[f'{var2}_bias'].values.flatten()
        
        ax.scatter(ds_metrics['time'], rmse, label='RMSE',  marker='o', facecolors='none', edgecolors='SteelBlue')
        ax.scatter(ds_metrics['time'], mae, label='MAE',  marker='.', c='seagreen')
        ax.scatter(ds_metrics['time'], bias, label='bias',  marker='o', facecolors='none', edgecolors='darkorange')


        ax.set_title(f'{var1} vs {var2}')
        ax.set_xlabel('time')
        ax.set_ylabel('error')
        ax.xaxis.set_major_locator(MonthLocator())
        ax.xaxis.set_major_formatter(DateFormatter('%b'))
        ax.xaxis.minorticks_off()
        ax.legend()

    fig.savefig(os.path.join(save_dir, f"{j}_tc_metrics.png"), dpi=100, bbox_inches='tight')
    plt.close(fig)


In [ ]:
# %%time

# for j, group in variable_pairs.items():
#     fig, axs = plt.subplots(nrows=3, ncols=1, figsize=(6,9), layout='constrained')

#     for i, (var1, var2) in enumerate(group):
#         ax=axs[i]

#         rmse = ds_metrics_monthly[f'{var2}_RMSE']
#         mae = ds_metrics_monthly[f'{var2}_MAE']
#         bias = ds_metrics_monthly[f'{var2}_bias']

#         ax.scatter(ds_metrics_monthly['month'], rmse, label='RMSE',  marker='o', facecolors='none', edgecolors='SteelBlue')
#         ax.scatter(ds_metrics_monthly['month'], mae, label='MAE',  marker='.', c='seagreen')
#         ax.scatter(ds_metrics_monthly['month'], bias, label='bias',  marker='o', facecolors='none', edgecolors='darkorange')


#         ax.set_title(f'{var1} vs {var2}')
#         ax.set_xlabel('time')
#         ax.set_ylabel('error')
#         ax.xaxis.set_major_locator(MonthLocator())
#         ax.xaxis.set_major_formatter(DateFormatter('%b'))
#         ax.xaxis.minorticks_off()
#         ax.legend()

#     fig.savefig(os.path.join(save_dir, f"{j}_tc_metrics_monthly.png"), dpi=100, bbox_inches='tight')
#     plt.close()


In [ ]:
# # original, don't touch. These boxplots are taking way too long to generate so I am skipping them for now.
# %%time

# for j, group in variable_pairs.items():
#     fig, axs = plt.subplots(nrows=3, ncols=1, figsize=(10,10), layout='constrained')

#     for i, (var1, var2) in enumerate(group):
#         colors = ['darkblue', 'peachpuff']
#         ax=axs[i]

#         months = []
#         positions = []
#         data = []
#         labels = []

#         for month, group in monthly_groups:
#             months.append(month)

#             data1 = group[var1].values.ravel()
#             data2 = group[var2].values.ravel()

#             mask = ~np.isnan(data1) & ~np.isnan(data2)
#             data1_clean = data1[mask]
#             data2_clean = data2[mask]

#             data.append(data1_clean)
#             data.append(data2_clean)

#             labels.append(f'{var1}')
#             labels.append(f'{var2}')

#             positions.append(month - 0.15)
#             positions.append(month + 0.15)

#         box_plot = ax.boxplot(data, positions=positions, widths=0.25, patch_artist=True, showfliers=False)

#         for patch, color in zip(box_plot['boxes'], colors*12):
#             patch.set_facecolor(color)

#         ax.set_title(f'{var1} vs {var2}')
#         ax.set_ylim(-0.6, 1)

#         ax.set_ylabel('vals')
#         ax.set_xlabel('Month')
#         ax.xaxis.set_major_locator(MonthLocator())
#         ax.xaxis.set_major_formatter(DateFormatter('%b'))
#         ax.xaxis.minorticks_off()

#         legend_patches = [plt.Line2D([0], [0], color=colors[0], lw=4, label=var1), plt.Line2D([0], [0], color=colors[1], lw=4, label=var2)]
#         ax.legend(handles=legend_patches, loc='upper right')
#         ax.set_xlim(0.5, 12.5)
#         #ax.tick_params(axis='x', rotation=70)

#     fig.suptitle('Monthly boxplots for Tasseled cap variable pairs for Landsat 8', fontsize=14)

#     fig.savefig(os.path.join(save_dir, f"{j}_tc_boxplots.png"), dpi=100, bbox_inches='tight')
#     plt.close()


In [ ]:
colors = ("indigo4", "green4", "yellow4")

for group_name, group in variable_pairs.items():
    group_start = time.time()
    fig, axs = plt.subplots(nrows=3, ncols=5, figsize=(15, 12), layout="constrained")

    for i, ((var1, var2), color) in enumerate(zip(group, colors)):
        step_start = time.time()

        # --- Phase 1: Prepare and clean data ---
        prep_start = time.time()
        data1 = ds_tc[var1].stack(space=("x", "y"))
        data2 = ds_tc[var2].stack(space=("x", "y"))

        valid_mask = (~np.isnan(data1)) & (~np.isnan(data2))
        valid_mask = valid_mask.compute()

        data1_clean = data1.where(valid_mask, drop=True).compute()
        data2_clean = data2.where(valid_mask, drop=True).compute()
        prep_end = time.time()

        # --- Phase 2: Axis limits ---
        limits_start = time.time()
        lower_limit = np.min((data1_clean.min(), data2_clean.min())) - 0.05
        upper_limit = np.max((data1_clean.max(), data2_clean.max())) + 0.05
        limits_end = time.time()

        # --- Phase 3: Precompute class values ---
        classmask_start = time.time()
        class_values_dict = {
            class_name: class_mask.stack(space=("x", "y"))
            .where(valid_mask, drop=True)
            .compute()
            .values
            for class_name, class_mask in class_masks.items()
        }
        classmask_end = time.time()

        # --- Phase 4: Plotting ---
        plot_start = time.time()
        for j, (class_name, class_values) in enumerate(class_values_dict.items()):
            ax = axs[i, j]

            class_filter = class_values == 1
            x_vals = data1_clean.values[class_filter]
            y_vals = data2_clean.values[class_filter]

            hb = ax.hexbin(
                x_vals, y_vals, gridsize=50, cmap="plasma", mincnt=1, marginals=True
            )
            ax.set_xlim(lower_limit, upper_limit)
            ax.set_ylim(lower_limit, upper_limit)
            ax.plot(
                (lower_limit, upper_limit),
                (lower_limit, upper_limit),
                c="darkgreen",
                alpha=0.5,
                ls="--",
                label="1:1",
            )

            if len(x_vals) > 1:
                result = linregress(x_vals, y_vals)
                ax.plot(
                    x_vals,
                    result.intercept + result.slope * x_vals,
                    c="darkorange",
                    label="regression",
                    alpha=0.5,
                )
                ax.text(
                    0.02,
                    0.98,
                    f"p-val:{result.pvalue:.4f}",
                    fontsize=12,
                    ha="left",
                    va="top",
                    transform=ax.transAxes,
                )
                r_squared = result.rvalue**2
            else:
                r_squared = np.nan

            ax.set_title(f"{var1} vs {var2}\n{class_name} R-sqr: {r_squared:.2f}")
            ax.set_xlabel(var1)
            ax.set_ylabel(var2)
            handles, labels = ax.get_legend_handles_labels()
            fig.legend(handles, labels, loc="upper left", ncol=1)
        plot_end = time.time()

        step_end = time.time()
        print(
            f"[{var1} vs {var2}] Prep: {prep_end - prep_start:.2f}s | Limits: {limits_end - limits_start:.2f}s | "
            f"Class masks: {classmask_end - classmask_start:.2f}s | Plotting: {plot_end - plot_start:.2f}s | Total: {step_end - step_start:.2f}s"
        )

    fig.savefig(
        os.path.join(save_dir, f"{group_name}_tc_hexbin.png"),
        dpi=150,
        bbox_inches="tight",
    )
    group_end = time.time()
    print(
        f"Saved plot for group '{group_name}' in {group_end - group_start:.2f} seconds"
    )
    plt.close(fig)


## Get TC percentiles and compare difference between variable pairs at 10, 50 and 90pc

- Use the `xr_quantile_bands` function to get the yearly 10th, 50th and 90th percentiles for the tasseled capped components generated with the current set of coefficients, and the proposed coefficients (drop unneeded data variables first)
- For each tasseled cap pair (TCW and TCW_new) and each percentile (10, 50, 90) calculate the delta and normalise the values between [-1, 1] so the difference between the tassel cap coefficients can be mapped
- threshold the delta values to see which areas are the most different between tassel caps derived from the different coefficients
- export summary `csv` of this to use in report

In [ ]:
def reduce(ds):
    yy = xr_quantile_bands(ds, [0.1, 0.5, 0.9], nodata=-9999)
    return yy


In [ ]:
ds_tc_percentiles_vars = ["TCW", "TCG", "TCB", "TCW_ls8ls9", "TCG_ls8ls9", "TCB_ls8ls9"]

ds_tc_prep = ds_tc.drop_vars(
    [var for var in ds_tc if var not in ds_tc_percentiles_vars + ["time", "x", "y"]]
)


In [ ]:
ds_tc_pc = reduce(ds_tc_prep)

ds_tc_pc


In [ ]:
percentiles = ["10", "50", "90"]

for i, (var1, var2) in enumerate(variable_pairs):
    for p in percentiles:
        var1_pc = f"{var1}_pc_{p}"
        var2_pc = f"{var2}_pc_{p}"

        if var1_pc in ds_tc_pc and var2_pc in ds_tc_pc:
            delta = ds_tc_pc[var1_pc] - ds_tc_pc[var2_pc]
            min_val = delta.min()
            max_val = delta.max()

            if max_val != min_val:
                normalised_delta = 2 * ((delta - min_val) / (max_val - min_val)) - 1
            else:
                normalised_delta = delta * 0

            ds_tc_pc[f"{var1_pc}_normalised_delta"] = normalised_delta
        else:
            print(f"Warning: {var1_pc} or {var2_pc} not found in dataset.")


In [ ]:
for i, (var1, var2) in enumerate(variable_pairs):
    for p in percentiles:
        var1_pc = f"{var1}_pc_{p}"
        var2_pc = f"{var2}_pc_{p}"

        if var1_pc in ds_tc_pc and var2_pc in ds_tc_pc:
            with np.errstate(divide="ignore", invalid="ignore"):
                percent_diff = (ds_tc_pc[var1_pc] - ds_tc_pc[var2_pc]) / ds_tc_pc[
                    var2_pc
                ]
                percent_diff = percent_diff.where(np.isfinite(percent_diff), 0)

                ds_tc_pc[f"{var1_pc}_percent_diff"] = percent_diff
        else:
            print(f"Warning: {var1_pc} or {var2_pc} not found in dataset.")


In [ ]:
ds_tc_pc.TCW_pc_50_percent_diff.plot(robust=True, cmap="RdBu_r")


In [ ]:
# apply threshold to TC delta

thresholds = (0.1, -0.1)

delta_threshold = ds_tc_pc.TCW_pc_50_normalised_delta.where(
    (ds_tc_pc.TCW_pc_50_normalised_delta > thresholds[0])
    | (ds_tc_pc.TCW_pc_50_normalised_delta < thresholds[1])
)

delta_threshold.plot(
    cmap="viridis",
    cbar_kwargs={"label": "TCW PC 50 Delta"},
)


In [ ]:
# data_vars = ["TCW_pc_10_delta", "TCG_pc_10_delta", "TCB_pc_10_delta"]

# fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(18, 6), layout="constrained")

# for ax, var in zip(axes, data_vars):
#     ds_tc_pc[var].plot(ax=ax, cmap="viridis")
#     ax.set_title(var)
#     ax.set_xlabel("Longitude")
#     ax.set_ylabel("Latitude")
#     ax.set_aspect("auto")
